# Chat with nano-dsv4.1f (T4 GPU or CPU)

Choose **GPU T4** (recommended; CPU also works, more slowly), enable **Internet**,
and attach the model `xiayicheng3gmailcom/nanodsv4-1f` (variation `chat-dspark`)
or your own bundle exported by `nano_dsv41f_export_sft_safetensors_cpu.ipynb`.
No optimizer, training corpus, or separate tokenizer input is needed.

The runtime comes from GitHub and uses the project's Torch implementation with
bounded caches, prefix reuse between turns, and DeepSeek V4.1 chat encoding.
`DEVICE = 'auto'` uses the GPU when one is attached. Inference runs in FP32; the saved
weights keep their original precision. The stored 32K context limit does not promise
fast long replies, so start with short prompts and outputs.
This notebook loads your trained weights; it never initializes a replacement model.


In [ ]:
SOURCE_REF = 'main'
MODEL_DIR = ''  # blank: locate exactly one exported SFT bundle under /kaggle/input
DEVICE = 'auto'  # 'auto' = cuda when a GPU is attached, else cpu
PREFILL_CHUNK_SIZE = 32
# DSpark speculative decoding verifies greedy decoding only. With a trained draft head on a
# T4 it measured 1.00-1.39x faster (mean ~1.15x); on CPU it measured slower. Greedy replies
# from this small model loop far more than sampled ones (T=0.7), so chat defaults to
# sampling. Set 'auto' (GPU + trained draft) or True to try the speed-up.
USE_MTP = False
# Decoding-time repetition control (DRY, "don't repeat yourself"): extending a span that already
# occurred in the reply costs 2.0 * 1.75**(length - 2) logits. On 32 fixed chat prompts it cut
# sampled replies that fall into a loop from 94% to 6% without changing the stop rate
# (docs/experiments/2026-11-10-decoding-repetition-controls.md). Set to None to disable.
REPETITION = {'dry_multiplier': 2.0, 'dry_base': 1.75, 'dry_allowed_length': 2}


In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

if any(name in sys.modules for name in ('jax', 'nano_dsv41f')):
    raise RuntimeError('Restart the session before updating the source and dependencies')
os.environ['JAX_PLATFORMS'] = 'cpu'
REPO = Path('/kaggle/working/nano-dsv4.1f')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/xiayicheng3-code/nano-dsv4.1f.git', str(REPO)], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain',
                            '--untracked-files=no'], text=True).strip():
    raise RuntimeError('Preserve tracked edits before updating this checkout')
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', SOURCE_REF], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
if importlib.util.find_spec('torch') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                    '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO / 'requirements-posttrain-inference.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), '--no-deps'], check=True)
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO / 'scripts'))
print('Source commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import json
import torch
from nano_dsv41f.portable_bundle import verify_portable_bundle
from nano_dsv41f.vllm_v41_cpu.api import NanoDeepSeekProtocolBackend

if MODEL_DIR:
    model_root = Path(MODEL_DIR)
else:
    bundles = list(Path('/kaggle/input').rglob('export_manifest.json'))
    if len(bundles) != 1:
        raise ValueError('Attach one exported bundle or set MODEL_DIR to its directory')
    model_root = bundles[0].parent
report = verify_portable_bundle(model_root)
config = json.loads((model_root / 'config.json').read_text())
MAX_CONTEXT = int(config['max_position_embeddings'])
torch.set_num_threads(max(1, (os.cpu_count() or 2) - 1))
if DEVICE == 'auto':
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
draft_trained = report.get("dspark_steps", 0) > 0
if USE_MTP == 'auto':
    USE_MTP = DEVICE == 'cuda' and draft_trained
print('Device:', DEVICE, '| DSpark speculative decoding:', USE_MTP,
      f'(draft trained for {report.get("dspark_steps", 0)} updates)')
backend = NanoDeepSeekProtocolBackend.from_pretrained(model_root, dtype=torch.float32, device=DEVICE)
from nano_dsv41f.vllm_v41_cpu.session import InferenceSession
backend.session = InferenceSession(backend.model, mtp=USE_MTP,
    draft_trained=draft_trained, prefill_chunk_size=PREFILL_CHUNK_SIZE)
if REPETITION:
    # DRY matches do not extend across newlines, colons, quotes or '*' (single tokens only).
    encoded = [backend.tokenizer.encode(t) for t in ('\n', '\n\n', ':', '"', '*')]
    backend.repetition = {**REPETITION, 'dry_sequence_breakers': sorted({e[0] for e in encoded if len(e) == 1})}
print('Loaded SFT checkpoint:', report['checkpoint'])
print('Repetition control:', backend.repetition)
print('Context:', MAX_CONTEXT, 'RoPE:', config['nano_config']['attention']['rope'])

def check_context(prompt_ids, max_tokens):
    if max_tokens <= 0 or len(prompt_ids) + max_tokens > MAX_CONTEXT:
        raise ValueError(f'Prompt + output must fit {MAX_CONTEXT} tokens; shorten the request or reset_chat()')

def complete(prompt, *, max_tokens=64, temperature=0.0):
    check_context(backend.tokenizer.encode(prompt), max_tokens)
    return backend.complete_raw(prompt, max_tokens=max_tokens, temperature=temperature)


In [ ]:
# Multi-turn DeepSeek V4.1 Chat Completions helper.
import json

history = []
REASONING_EFFORT_PRESETS = {"low": 50, "high": 75, "max": 100}

def reset_chat():
    history.clear()
    backend.reset_cache()

def chat(message, *, thinking=False, reasoning_effort="high", max_tokens=128, temperature=None):
    if thinking:
        print("Warning: thinking mode and reasoning_effort are not functional on this "
              "checkpoint; replies usually never leave the reasoning block.")
    if isinstance(reasoning_effort, str):
        if reasoning_effort not in REASONING_EFFORT_PRESETS:
            raise ValueError("reasoning_effort must be low, high, max, or an integer in [1, 100]")
        reasoning_effort = REASONING_EFFORT_PRESETS[reasoning_effort]
    if type(reasoning_effort) is not int or not 1 <= reasoning_effort <= 100:
        raise ValueError("reasoning_effort must be low, high, max, or an integer in [1, 100]")
    if temperature is None:
        temperature = 0.0 if getattr(getattr(backend, "session", None), "mtp", False) else 0.7
    request_messages = history + [{"role": "user", "content": message}]
    payload = {
        "model": backend.model_name,
        "messages": request_messages,
        "thinking": {"type": "enabled" if thinking else "disabled"},
        "reasoning_effort": reasoning_effort,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": 0.9,
        "stream": False,
    }
    from nano_dsv41f.vllm_v41_cpu.api import prepare_protocol_request
    prepared = prepare_protocol_request("chat_completions", payload)
    prompt_ids = backend.tokenizer.encode_request(prepared)
    check_context(prompt_ids, max_tokens)
    _, raw = backend.complete_protocol("chat_completions", payload)
    response = json.loads(raw)
    assistant = response["choices"][0]["message"]
    history.append({"role": "user", "content": message})
    saved = {"role": "assistant", "content": assistant.get("content") or ""}
    if assistant.get("reasoning_content"):
        saved["reasoning_content"] = assistant["reasoning_content"]
    if assistant.get("tool_calls"):
        saved["tool_calls"] = assistant["tool_calls"]
    history.append(saved)
    assistant["_inference_stats"] = dict(getattr(backend, "last_stats", {}))
    return assistant

def print_reply(reply):
    reasoning = reply.get("reasoning_content")
    if reasoning:
        print("Reasoning:\n" + reasoning + "\n")
    print("Assistant:\n" + (reply.get("content") or ""))
    from nano_dsv41f.vllm_v41_cpu.session import format_inference_stats
    print(format_inference_stats(reply.get("_inference_stats", {})))


In [ ]:
# A short first response checks model loading, chat encoding, and cached generation.
print_reply(chat('Hello! Introduce yourself in one sentence.', max_tokens=32))
reset_chat()


## Chat here

The next cell creates a small in-notebook chat control. Type a message and press
**Send**; the conversation history is preserved until you press **Reset**.
Input, Send, Reset, and Max tokens are locked while a response is being generated.

Replies are sampled (temperature 0.7, top-p 0.9) with a DRY repetition penalty
(`REPETITION` in the first cell; set it to `None` to see the raw model). Thinking mode and reasoning effort
are not offered: on this checkpoint, thinking-mode replies did not finish their
reasoning or answer at any effort level, so chat always uses non-thinking mode.


In [ ]:
# Kaggle/Jupyter-native chat controls: no public tunnel or separate web app needed.
import asyncio
import ipywidgets as widgets
from IPython.display import display

if globals().get("_chat_task") is not None and not _chat_task.done():
    raise RuntimeError("Wait for the current response before rebuilding chat controls")
message_box = widgets.Textarea(
    placeholder="Type a message…",
    layout=widgets.Layout(width="100%", height="90px"),
)
max_tokens_box = widgets.BoundedIntText(
    value=128,
    min=1,
    max=1024,
    step=1,
    description="Max tokens:",
)
send_button = widgets.Button(description="Send", button_style="primary")
reset_button = widgets.Button(description="Reset")
chat_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd"))
status_label = widgets.Label(value="Ready")
_chat_busy = False
_chat_task = None

def _set_busy(busy):
    global _chat_busy
    _chat_busy = busy
    for control in (message_box, max_tokens_box, send_button, reset_button):
        control.disabled = busy
    send_button.description = "Generating…" if busy else "Send"
    status_label.value = "Generating response…" if busy else "Ready"

async def _generate_reply(message, settings):
    try:
        # Keep the kernel event loop available for widget updates and
        # rejecting extra clicks. All widget writes stay on this loop.
        reply = await asyncio.to_thread(chat, message, **settings)
        reasoning = reply.get("reasoning_content")
        if reasoning:
            chat_output.append_stdout(f"Reasoning: {reasoning}\n")
        chat_output.append_stdout(f"Assistant: {reply.get('content') or ''}\n")
        from nano_dsv41f.vllm_v41_cpu.session import format_inference_stats
        chat_output.append_stdout(format_inference_stats(reply.get("_inference_stats", {})) + "\n\n")
    except Exception as exc:
        message_box.value = message
        chat_output.append_stdout(f"Error: {exc}\n\n")
    finally:
        _set_busy(False)

def _send(_):
    global _chat_task
    if _chat_busy:
        return
    message = message_box.value.strip()
    if not message:
        return
    settings = dict(max_tokens=max_tokens_box.value)
    _set_busy(True)
    message_box.value = ""
    chat_output.append_stdout(f"You: {message}\n")
    try:
        _chat_task = asyncio.get_running_loop().create_task(_generate_reply(message, settings))
    except Exception as exc:
        message_box.value = message
        _set_busy(False)
        chat_output.append_stdout(f"Error: {exc}\n\n")

def _reset(_):
    if _chat_busy:
        return
    reset_chat()
    chat_output.clear_output()
    chat_output.append_stdout("Conversation reset.\n\n")

send_button.on_click(_send)
reset_button.on_click(_reset)
display(
    widgets.VBox(
        [
            message_box,
                widgets.HBox([max_tokens_box, send_button, reset_button]),
            status_label,
            chat_output,
        ]
    )
)


## Or call the helper directly

```python
print_reply(chat("Why is sparse attention useful?"))
print_reply(chat("Now explain that more simply."))  # keeps the same history
```

For DeepSeek V4.1 reasoning mode, use `low` (50), `high` (75, the default),
`max` (100), or any integer in `[1, 100]`, matching the open-weight encoder:

```python
print_reply(
    chat(
        "Solve 37*43 carefully.",
        thinking=True,
        reasoning_effort="high",
        max_tokens=256,
    )
)
print_reply(chat("Try a shorter explanation.", thinking=True, reasoning_effort=30))
```

Use `reset_chat()` to start a fresh conversation.


## Raw completion and tool output

`complete("The capital of France is", max_tokens=32)` uses the prompt directly.
`chat(...)` applies the V4.1 conversation protocol and maintains history.
Both functions check prompt plus requested output against the saved context limit.
For tool-call inspection, examine the returned reply dictionary; this notebook
does not execute generated tool calls. Use `reset_chat()` for a new conversation.
Every reply reports prefill/decode token counts, elapsed seconds, tokens/second,
reused prompt tokens, first-token latency, and reserved cache arena size.
Prefill throughput counts only newly processed tokens. Decode counts include
reasoning, protocol tokens, and EOS when generated; its time includes sampling
and any drafting. The cache arena figure excludes weights, temporary tensors,
and the small prompt checkpoint. Detailed step times are in `backend.last_stats`.

Prefix caches persist in session RAM until reset/restart. When the protocol
rewrites a previous reply, the runtime reuses the last matching prompt checkpoint
or rebuilds safely. Prompt prefill batches up to `PREFILL_CHUNK_SIZE` tokens
per target pass; set it to 1 for the scalar comparison. No fused GPU kernels
or independent-request batching are used.

To accelerate greedy decoding, attach the bundle from the DSpark distillation
notebook and set `USE_MTP=True`. The verified export must record completed
draft updates. Chat automatically uses temperature zero in this mode.
One causal target pass verifies a proposal block. The first mismatch is
replaced with the target token, and rejected cache entries are rolled back
without recomputing the accepted prefix. Output matches greedy target decoding
within normal floating-point limits. Speed depends on acceptance and hardware;
an inaccurate drafter can still make decoding slower.

`backend.last_stats` includes target calls/input tokens, drafting time,
verification time, rollback time, and accepted/verified/proposed counts.
Batch times and emitted counts are recorded separately; per-token step times
are amortized within each batch. Set `USE_MTP=False` to compare normal decoding.
For a real-bundle benchmark, run `scripts/benchmark_mtp_inference.py` with
`--model-dir`, `--device cpu` or `cuda`, and `--output /path/to/results.json`.
The optional local API cell below serves the same loaded model.


In [ ]:
# Optional: expose the same model through local OpenAI/Anthropic-compatible HTTP routes.
# Run this cell only when you want an API server inside the Kaggle session.
# from nano_dsv41f.vllm_v41_cpu.api import create_app
# import uvicorn
# app = create_app(backend)
# uvicorn.run(app, host="127.0.0.1", port=8000)
